# MaNGA dataloader preview

Inspect `MangaGalaxyDataset` samples before UNet training: **input images**, optional spectrum, Amara map targets, and masks.

See [`docs/manga_dataloader.md`](docs/manga_dataloader.md) for full API documentation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from manga_prep.targets.pipe3d_maps import AMARA_TARGET_KEYS
from manga_prep.dataset.manga_dataset import MangaGalaxyDataset

DATA_ROOT = Path("manga_sdss_fits")
INDEX_PATH = DATA_ROOT / "manga_dataset_index.csv"

TARGET_LABELS = {
    "ha_flux": "Hα flux",
    "hbeta_flux": "Hβ flux",
    "oiii_5007_flux": "[OIII]5007",
    "nii_6584_flux": "[NII]6584",
    "ha_ew": "Hα EW",
    "stellar_av": "Stellar Av",
}

In [ ]:
# Toggle which inputs to load (init is slow when SDSS/Legacy imaging is enabled)
INCLUDE_SDSS = True
INCLUDE_LEGACY = False
SPECTRUM = "fake"  # None | "real" | "fake"
ALIGN_IMAGING = True  # reproject cutouts onto Pipe3D / Amara spaxel WCS (76x76)

dataset = MangaGalaxyDataset(
    DATA_ROOT,
    INDEX_PATH,
    include_sdss_imaging=INCLUDE_SDSS,
    include_legacy_imaging=INCLUDE_LEGACY,
    include_targets=True,
    spectrum=SPECTRUM,
    align_imaging_to_amara_grid=ALIGN_IMAGING,
    require_all=True,
)

print(f"Dataset size: {len(dataset):,} galaxies")
print(f"Target channels: {list(AMARA_TARGET_KEYS)}")


In [ ]:
# Pick a sample: random index, or set PLATEIFU = "7495-3702"
PLATEIFU = None  # e.g. "7495-3702"
RANDOM_SEED = 42

if PLATEIFU is not None:
    idx = next(i for i, row in enumerate(dataset.rows) if row["plateifu"] == PLATEIFU)
else:
    rng = np.random.default_rng(RANDOM_SEED)
    idx = int(rng.integers(0, len(dataset)))

sample = dataset[idx]
print(f"index={idx}  plateifu={sample['plateifu']}")
print(f"native_shape={sample['native_shape']}  target_shape={sample['target_shape']}")
print(f"footprint spaxels={sample['footprint_mask'].sum()} / {sample['footprint_mask'].size}")
print(f"inputs: {list(sample.get('inputs', {}).keys())}")

In [ ]:
def _percentile_norm(x, lo_pct=5, hi_pct=99):
    x = np.nan_to_num(x, nan=0.0)
    pos = x[x > 0]
    if pos.size == 0:
        return np.zeros_like(x)
    lo, hi = np.percentile(pos, [lo_pct, hi_pct])
    return np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)


def show_imaging_stack(title, stack, bands, *, rgb_bands=("r", "g", "i")):
    """Show every aligned input band plus RGB composite (76x76 spaxel grid)."""
    bidx = {b: i for i, b in enumerate(bands)}
    n = len(bands)
    fig, axes = plt.subplots(1, n + 1, figsize=(2.4 * (n + 1), 2.6))
    if n + 1 == 1:
        axes = [axes]

    for ax, band in zip(axes[:n], bands):
        img = stack[bidx[band]]
        im = ax.imshow(_percentile_norm(img), origin="lower", cmap="gray", vmin=0, vmax=1)
        ax.set_title(band)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    ax_rgb = axes[-1]
    if all(b in bidx for b in rgb_bands):
        r, g, b = (_percentile_norm(stack[bidx[band]]) for band in rgb_bands)
        rgb = np.dstack([r, g, b])
        ax_rgb.imshow(rgb, origin="lower")
        ax_rgb.set_title("rgb (" + "".join(rgb_bands) + ")")
    ax_rgb.axis("off")

    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()


def show_input_images(sample):
    inputs = sample.get("inputs", {})
    if not inputs:
        print("No inputs in sample")
        return

    aligned = " (aligned to Amara spaxel WCS)" if inputs.get("sdss_imaging", {}).get("aligned_to_amara_grid") else ""
    if "sdss_imaging" in inputs:
        show_imaging_stack(
            f"SDSS inputs{aligned} — {sample['plateifu']}",
            inputs["sdss_imaging"]["data"],
            inputs["sdss_imaging"]["bands"],
            rgb_bands=("r", "g", "i"),
        )

    if "legacy_imaging" in inputs:
        show_imaging_stack(
            f"Legacy inputs{aligned} — {sample['plateifu']}",
            inputs["legacy_imaging"]["data"],
            inputs["legacy_imaging"]["bands"],
            rgb_bands=("r", "g", "z"),
        )


def show_imaging_target_overlay(sample, band="r", target_key="ha_flux"):
    """Overlay aligned imaging on an Amara target map to verify orientation."""
    img_in = sample.get("inputs", {}).get("sdss_imaging")
    if img_in is None:
        print("Need SDSS imaging for overlay")
        return
    bidx = {b: i for i, b in enumerate(img_in["bands"])}
    if band not in bidx:
        band = img_in["bands"][0]
    image = _percentile_norm(img_in["data"][bidx[band]])
    target = sample["targets"][target_key]
    mask = sample["target_loss_masks"][target_key].astype(bool)
    target_masked = np.where(mask, target, np.nan)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image, origin="lower", cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"SDSS {band} (aligned)")
    im1 = axes[1].imshow(target_masked, origin="lower", cmap="viridis", vmin=0, vmax=1)
    axes[1].set_title(TARGET_LABELS.get(target_key, target_key))
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    axes[2].imshow(image, origin="lower", cmap="gray", vmin=0, vmax=1, alpha=0.55)
    axes[2].imshow(target_masked, origin="lower", cmap="magma", vmin=0, vmax=1, alpha=0.55)
    axes[2].set_title("overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"Orientation check — {sample['plateifu']}")
    plt.tight_layout()
    plt.show()


In [ ]:
show_input_images(sample)
show_imaging_target_overlay(sample, band="r", target_key="ha_flux")


In [ ]:
def show_spectrum(sample):
    spec = sample.get("inputs", {}).get("spectrum")
    if spec is None:
        print("No spectrum in sample")
        return
    wave = spec["wave"]
    flux = spec["flux"]
    good = np.isfinite(flux)
    label = "real SDSS" if spec["is_real_sdss_fiber"] else "fake SDSS"
    plt.figure(figsize=(8, 3))
    plt.plot(wave[good], flux[good], lw=0.8)
    plt.xlabel("Wavelength [Å]")
    plt.ylabel("Flux")
    plt.title(f"Input spectrum ({label}) — {sample['plateifu']}")
    plt.tight_layout()
    plt.show()

show_spectrum(sample)

In [ ]:
def show_target_maps(sample, keys=None):
    keys = list(keys or AMARA_TARGET_KEYS)
    n = len(keys)
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for j, key in enumerate(keys):
        tgt = sample["targets"][key]
        m = sample["target_loss_masks"][key].astype(bool)
        masked = np.where(m, tgt, np.nan)

        ax0, ax1 = axes[0, j], axes[1, j]
        im0 = ax0.imshow(tgt, origin="lower", cmap="viridis", vmin=0, vmax=1)
        ax0.set_title(f"{TARGET_LABELS.get(key, key)} target")
        ax0.axis("off")
        plt.colorbar(im0, ax=ax0, fraction=0.046)

        im1 = ax1.imshow(masked, origin="lower", cmap="viridis", vmin=0, vmax=1)
        ax1.set_title("loss mask applied")
        ax1.axis("off")
        plt.colorbar(im1, ax=ax1, fraction=0.046)

    fig.suptitle(f"Amara targets — {sample['plateifu']}", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_masks(sample):
    footprint = sample["footprint_mask"]
    valid_union = np.zeros_like(footprint, dtype=bool)
    loss_union = np.zeros_like(footprint, dtype=bool)
    for key in AMARA_TARGET_KEYS:
        valid_union |= sample["target_valid_masks"][key].astype(bool)
        loss_union |= sample["target_loss_masks"][key].astype(bool)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, arr, title in zip(
        axes,
        [footprint, valid_union, loss_union],
        ["SELECT_REG footprint", "any feature valid", "any loss mask"],
    ):
        ax.imshow(arr, origin="lower", cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{title}\n({arr.sum()} px)")
        ax.axis("off")
    fig.suptitle(f"Masks — {sample['plateifu']}")
    plt.tight_layout()
    plt.show()

show_target_maps(sample)
show_masks(sample)

In [ ]:
# Quick sanity: loss mask is subset of footprint
for key in AMARA_TARGET_KEYS:
    lm = sample["target_loss_masks"][key].astype(bool)
    fp = sample["footprint_mask"].astype(bool)
    assert np.all(lm <= fp), f"loss mask exceeds footprint for {key}"
    print(f"{key:16s} loss={lm.sum():4d}  valid={sample['target_valid_masks'][key].sum():4d}")
print("OK: all loss masks are inside footprint")